# RG-CBM: Radiomics-Guided Concept Bottleneck Model

This notebook implements ** RG-CBM model**:

```text
MRI
→ EfficientNetB1 feature map
→ 1×1 projection + L2 normalization
→ learnable concept prototypes + cosine matching
→ radiomics-guided auxiliary losses (boundary, texture, morphology)
→ global average pooling → concept vector
→ sparse linear classifier
→ exact concept contributions / interventions
→ automatic post-training concept grounding and naming
→ LLM doctor-understandable explanation from model evidence only
```


In [ ]:
# ============================================================
# 0. Setup
# ============================================================
import os, sys, math, json, random, shutil, subprocess, time, re
from pathlib import Path
from datetime import datetime
from getpass import getpass

# Gemini = convenient free-tier cloud LLM; Transformers = local no-API fallback.
!pip -q install -U kaggle kagglehub scikit-image opencv-python-headless scikit-learn pandas matplotlib google-genai transformers accelerate

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers

# Avoid TensorFlow reserving all GPU memory. This is especially useful if the
# optional local Hugging Face LLM is used later in the same Colab session.
for _gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(_gpu, True)
    except Exception:
        pass

import cv2
from skimage.filters import threshold_otsu
from skimage.measure import label, regionprops

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

from IPython.display import display, Markdown

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)


## 1. Runtime, Google Drive, and artifact directories

In [ ]:
# ============================================================
# 1. Runtime + Google Drive backup configuration
# ============================================================
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = Path("/kaggle/working").exists()
USE_GOOGLE_DRIVE = True

RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_NAME = f"RG_CBM_EfficientNetB1_{RUN_STAMP}"

if IN_COLAB:
    LOCAL_RUN_DIR = Path("/content/rg_cbm_work") / RUN_NAME
elif IN_KAGGLE:
    LOCAL_RUN_DIR = Path("/kaggle/working") / RUN_NAME
else:
    LOCAL_RUN_DIR = Path("./rg_cbm_work") / RUN_NAME
LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = LOCAL_RUN_DIR

DRIVE_RUN_DIR = None
if IN_COLAB and USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_RUN_DIR = Path("/content/drive/MyDrive/RG_CBM_Results") / RUN_NAME
        DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
        print("Google Drive backup:", DRIVE_RUN_DIR)
    except Exception as exc:
        print("Google Drive could not be mounted; continuing with local outputs.")
        print("Reason:", repr(exc))
else:
    print("Google Drive auto-sync is available in Colab. Local output directory:", RUN_DIR)


def sync_selected_to_drive(relative_paths=None, sync_all=False):
    """Copy selected artifacts (or the complete run folder) to Google Drive."""
    if DRIVE_RUN_DIR is None:
        return

    if sync_all:
        for src in RUN_DIR.rglob("*"):
            if src.is_file():
                rel = src.relative_to(RUN_DIR)
                dst = DRIVE_RUN_DIR / rel
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src, dst)
        return

    for rel in (relative_paths or []):
        src = RUN_DIR / rel
        if src.exists() and src.is_file():
            dst = DRIVE_RUN_DIR / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)

print("RUN_DIR:", RUN_DIR)


## 2. Kaggle dataset

This notebook uses the dataset from URL:

`https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset`


In [ ]:
# ============================================================
# 2. Dataset setup
# ============================================================
DATASET_URL = "https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset"
DATASET_SLUG = "masoudnickparvar/brain-tumor-mri-dataset"
FORCE_DATASET_REFRESH = True   # set False after the first successful download if desired

EXPECTED_TRAIN_PER_CLASS = 1400
EXPECTED_TEST_PER_CLASS = 400
EXPECTED_CLASSES = {"glioma", "meningioma", "notumor", "pituitary"}

if IN_COLAB:
    REQUESTED_DATASET_DIR = Path("/content/rg_cbm_requested_kaggle_dataset")
elif IN_KAGGLE:
    REQUESTED_DATASET_DIR = Path("/kaggle/working/rg_cbm_requested_kaggle_dataset")
else:
    REQUESTED_DATASET_DIR = Path("./rg_cbm_requested_kaggle_dataset")

MANUAL_TRAIN_DIR = None
MANUAL_TEST_DIR = None

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tif", ".tiff"}


def normalize_name(name: str) -> str:
    return str(name).lower().replace(" ", "").replace("_", "").replace("-", "")


def count_images(folder: Path) -> int:
    folder = Path(folder)
    if not folder.exists():
        return 0
    return sum(1 for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)


def class_subdirs(split_dir: Path):
    split_dir = Path(split_dir)
    if not split_dir.exists():
        return []
    return sorted(
        [p for p in split_dir.iterdir() if p.is_dir() and count_images(p) > 0],
        key=lambda p: p.name.lower(),
    )


def find_child_by_alias(parent: Path, aliases):
    aliases = {normalize_name(a) for a in aliases}
    if not Path(parent).exists():
        return None
    for child in Path(parent).iterdir():
        if child.is_dir() and normalize_name(child.name) in aliases:
            return child
    return None


def locate_dataset_splits(search_root: Path):
    search_root = Path(search_root)
    if not search_root.exists():
        return None, None, None
    candidates = [search_root] + [p for p in search_root.rglob("*") if p.is_dir()]
    for root in candidates:
        train_dir = find_child_by_alias(root, ["Training", "Train"])
        test_dir = find_child_by_alias(root, ["Testing", "Test"])
        if train_dir is not None and test_dir is not None:
            if count_images(train_dir) > 0 and count_images(test_dir) > 0:
                return root, train_dir, test_dir
    return None, None, None


def download_requested_dataset():
    """Download the exact Kaggle slug. No generic local auto-detection."""
    print("Requested dataset:", DATASET_URL)
    print("Kaggle slug:", DATASET_SLUG)
    print("Force refresh:", FORCE_DATASET_REFRESH)

    try:
        import kagglehub
        REQUESTED_DATASET_DIR.mkdir(parents=True, exist_ok=True)
        downloaded = Path(
            kagglehub.dataset_download(
                DATASET_SLUG,
                force_download=FORCE_DATASET_REFRESH,
                output_dir=str(REQUESTED_DATASET_DIR),
            )
        )
        print("KaggleHub returned:", downloaded)
        return downloaded
    except Exception as hub_exc:
        print("KaggleHub download failed:", repr(hub_exc))
        print("Trying Kaggle CLI into the dedicated requested-dataset folder...")

    try:
        if FORCE_DATASET_REFRESH and REQUESTED_DATASET_DIR.exists():
            shutil.rmtree(REQUESTED_DATASET_DIR, ignore_errors=True)
        REQUESTED_DATASET_DIR.mkdir(parents=True, exist_ok=True)
        subprocess.run(
            ["kaggle", "datasets", "download", "-d", DATASET_SLUG,
             "-p", str(REQUESTED_DATASET_DIR), "--unzip"],
            check=True,
        )
        return REQUESTED_DATASET_DIR
    except Exception as cli_exc:
        raise RuntimeError(
            "Could not download the requested Kaggle dataset. In Colab, add your "
            "KAGGLE_API_TOKEN/kaggle.json if Kaggle asks for authentication. "
            f"KaggleHub/Kaggle CLI error: {cli_exc}"
        )


if MANUAL_TRAIN_DIR and MANUAL_TEST_DIR:
    DATASET_ROOT = Path(MANUAL_TRAIN_DIR).parent
    TRAIN_DIR = Path(MANUAL_TRAIN_DIR)
    TEST_DIR = Path(MANUAL_TEST_DIR)
    DATASET_SOURCE = "manual_override"
else:
    downloaded_root = download_requested_dataset()
    DATASET_ROOT, TRAIN_DIR, TEST_DIR = locate_dataset_splits(downloaded_root)
    DATASET_SOURCE = DATASET_SLUG

if TRAIN_DIR is None or TEST_DIR is None:
    searched_root = DATASET_ROOT if (MANUAL_TRAIN_DIR and MANUAL_TEST_DIR) else downloaded_root
    raise FileNotFoundError(
        "The exact requested Kaggle dataset was downloaded/attached, but Training/Testing "
        f"folders could not be found inside {searched_root}."
    )

train_counts = {normalize_name(p.name): count_images(p) for p in class_subdirs(TRAIN_DIR)}
test_counts = {normalize_name(p.name): count_images(p) for p in class_subdirs(TEST_DIR)}
actual_classes = set(train_counts)
actual_test_classes = set(test_counts)

print("\n===== DATASET PROVENANCE =====")
print("Dataset URL:", DATASET_URL)
print("Dataset root:", DATASET_ROOT)
print("Training dir:", TRAIN_DIR)
print("Testing dir:", TEST_DIR)
print("Training counts:", train_counts, "total=", sum(train_counts.values()))
print("Testing counts:", test_counts, "total=", sum(test_counts.values()))

if actual_classes != EXPECTED_CLASSES or actual_test_classes != EXPECTED_CLASSES:
    raise RuntimeError(
        f"Unexpected classes. training={actual_classes}, testing={actual_test_classes}, "
        f"expected={EXPECTED_CLASSES}"
    )

expected_train = {k: EXPECTED_TRAIN_PER_CLASS for k in EXPECTED_CLASSES}
expected_test = {k: EXPECTED_TEST_PER_CLASS for k in EXPECTED_CLASSES}
counts_match_current_kaggle = (train_counts == expected_train and test_counts == expected_test)

manifest = {
    "dataset_url": DATASET_URL,
    "dataset_slug": DATASET_SLUG,
    "dataset_source": DATASET_SOURCE,
    "dataset_root": str(DATASET_ROOT),
    "train_dir": str(TRAIN_DIR),
    "test_dir": str(TEST_DIR),
    "force_dataset_refresh": FORCE_DATASET_REFRESH,
    "training_counts": train_counts,
    "testing_counts": test_counts,
    "total_images": int(sum(train_counts.values()) + sum(test_counts.values())),
    "matches_expected_current_kaggle_counts": bool(counts_match_current_kaggle),
    "timestamp": datetime.now().isoformat(),
}
with open(RUN_DIR / "dataset_provenance_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)
sync_selected_to_drive(["dataset_provenance_manifest.json"])


In [ ]:
# ============================================================
# 3. Configuration
# ============================================================
IMG_SIZE = (240, 240)
BATCH_SIZE = 24
VAL_SPLIT = 0.10

NUM_CONCEPTS = 16
NUM_BOUNDARY_CONCEPTS = 4
NUM_TEXTURE_CONCEPTS = 4
NUM_MORPHOLOGY_CONCEPTS = 4
NUM_UNGUIDED_CONCEPTS = 4
assert NUM_BOUNDARY_CONCEPTS + NUM_TEXTURE_CONCEPTS + NUM_MORPHOLOGY_CONCEPTS + NUM_UNGUIDED_CONCEPTS == NUM_CONCEPTS

PROTOTYPE_DIM = 128

EPOCHS_HEAD = 20
EPOCHS_FINETUNE = 20
HEAD_LR = 4e-4
FINETUNE_LR = 8e-6
FINE_TUNE_LAST_N_LAYERS = 90

AUX_CLASSIFICATION_ONLY_EPOCHS = 3
AUX_RAMP_EPOCHS = 5
AUX_TARGET_SCALE = 1.0

LAMBDA_EDGE = 0.18
LAMBDA_TEXTURE = 0.10
LAMBDA_MORPH = 0.08
LAMBDA_DIVERSITY = 0.02
LAMBDA_DISENTANGLE = 0.01
LAMBDA_CONCEPT_SPARSE = 5e-5
LAMBDA_STABILITY = 0.02
CLASSIFIER_L1 = 5e-6
CONCEPT_DROPOUT = 0.10

# Milder MRI-safe augmentation .
AUG_ROTATION = 0.025
AUG_ZOOM = 0.04
AUG_TRANSLATION = 0.02
AUG_CONTRAST = 0.06

# ------------------------------------------------------------------
# LLM communication layer (prediction is already fixed before this).
# ------------------------------------------------------------------
USE_LLM_REPORTS = True

# Recommended options:
#   "gemini"      -> cloud, Gemini 2.5 Flash, currently has a free tier
#   "local_qwen"  -> fully local, no API key, slower on CPU
#   "none"        -> generate evidence JSON only
LLM_BACKEND = os.getenv("RGCBM_LLM_BACKEND", "gemini").lower()
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")
LOCAL_LLM_MODEL = os.getenv("LOCAL_LLM_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
LOCAL_LLM_DEVICE = os.getenv("LOCAL_LLM_DEVICE", "cpu")  # use "cuda" only if memory is sufficient
ALLOW_LOCAL_FALLBACK = True
BATCH_LLM_CASES = 8

print("Configuration loaded.")
print("LLM backend:", LLM_BACKEND)


In [ ]:
# ============================================================
# 4. tf.data loading
# ============================================================
AUTOTUNE = tf.data.AUTOTUNE

train_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="int",
    color_mode="rgb",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
    validation_split=VAL_SPLIT,
    subset="training",
)

val_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="int",
    color_mode="rgb",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
    validation_split=VAL_SPLIT,
    subset="validation",
)

test_raw = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    labels="inferred",
    label_mode="int",
    color_mode="rgb",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

class_names = train_raw.class_names
num_classes = len(class_names)
print("Class names:", class_names)


def keep_float_images(x, y):
    return tf.cast(x, tf.float32), tf.cast(y, tf.int32)

train_ds = train_raw.map(keep_float_images, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds = val_raw.map(keep_float_images, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds = test_raw.map(keep_float_images, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)


def collect_labels(ds):
    ys = []
    for _, y in ds:
        ys.extend(y.numpy().tolist())
    return np.asarray(ys, dtype=np.int32)

train_labels_for_weights = collect_labels(train_raw)
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_classes),
    y=train_labels_for_weights,
)
class_weight = {int(i): float(w) for i, w in enumerate(weights)}
print("Class weights:", class_weight)

plt.figure(figsize=(12, 7))
for images, labels in train_ds.take(1):
    for i in range(min(12, images.shape[0])):
        ax = plt.subplot(3, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i])])
        plt.axis("off")
plt.tight_layout()
plt.show()


## 5. Radiomics-inspired priors

Boundary guidance uses Sobel + Laplacian responses. Texture guidance combines local variance with a differentiable local co-occurrence proxy based on neighboring gray-level contrast and homogeneity. Morphology guidance remains a pseudo-mask because no manual segmentation masks are required. Intensity is retained for **post-training grounding/naming**, not as a manually annotated concept target.


In [ ]:
# ============================================================
# 5. Differentiable radiomics-inspired weak supervision maps
# ============================================================
def normalize_per_image(x, eps=1e-8):
    x = tf.cast(x, tf.float32)
    min_v = tf.reduce_min(x, axis=[1, 2, 3], keepdims=True)
    max_v = tf.reduce_max(x, axis=[1, 2, 3], keepdims=True)
    return (x - min_v) / (max_v - min_v + eps)


def rgb_to_gray01(images):
    images = tf.cast(images, tf.float32)
    images = tf.clip_by_value(images, 0.0, 255.0)
    return tf.image.rgb_to_grayscale(images) / 255.0


def sobel_edge_map_tf(images):
    gray = rgb_to_gray01(images)
    sobel = tf.image.sobel_edges(gray)
    mag = tf.sqrt(tf.reduce_sum(tf.square(sobel), axis=-1) + 1e-8)
    return normalize_per_image(mag)


def log_edge_map_tf(images):
    gray = rgb_to_gray01(images)
    kernel = tf.constant([[0., 1., 0.], [1., -4., 1.], [0., 1., 0.]], dtype=tf.float32)
    kernel = tf.reshape(kernel, [3, 3, 1, 1])
    lap = tf.abs(tf.nn.conv2d(gray, kernel, strides=1, padding="SAME"))
    return normalize_per_image(lap)


def combined_edge_map_tf(images):
    return normalize_per_image(0.7 * sobel_edge_map_tf(images) + 0.3 * log_edge_map_tf(images))


def local_variance_texture_tf(images, ksize=9):
    gray = rgb_to_gray01(images)
    mean = tf.nn.avg_pool2d(gray, ksize=ksize, strides=1, padding="SAME")
    mean_sq = tf.nn.avg_pool2d(tf.square(gray), ksize=ksize, strides=1, padding="SAME")
    var = tf.nn.relu(mean_sq - tf.square(mean))
    return normalize_per_image(tf.sqrt(var + 1e-8))


def glcm_inspired_texture_tf(images, ksize=9):
    """
    Differentiable local proxy for GLCM-style contrast/homogeneity.
    It is intentionally called GLCM-inspired rather than an exact discrete GLCM.
    """
    gray = rgb_to_gray01(images)
    right = tf.pad(gray[:, :, 1:, :], [[0,0],[0,0],[0,1],[0,0]], mode="SYMMETRIC")
    down = tf.pad(gray[:, 1:, :, :], [[0,0],[0,1],[0,0],[0,0]], mode="SYMMETRIC")
    diff2 = 0.5 * (tf.square(gray - right) + tf.square(gray - down))
    local_contrast = tf.nn.avg_pool2d(diff2, ksize=ksize, strides=1, padding="SAME")
    local_homogeneity = tf.nn.avg_pool2d(1.0 / (1.0 + 16.0 * diff2), ksize=ksize, strides=1, padding="SAME")
    texture = 0.75 * normalize_per_image(local_contrast) + 0.25 * (1.0 - normalize_per_image(local_homogeneity))
    return normalize_per_image(texture)


def texture_map_tf(images):
    return normalize_per_image(0.75 * local_variance_texture_tf(images) + 0.25 * glcm_inspired_texture_tf(images))


def pseudo_morphology_mask_tf(images):
    gray = rgb_to_gray01(images)
    mean = tf.reduce_mean(gray, axis=[1, 2, 3], keepdims=True)
    std = tf.math.reduce_std(gray, axis=[1, 2, 3], keepdims=True)
    high = tf.cast(gray > (mean + 0.45 * std), tf.float32)
    low = tf.cast(gray < (mean - 0.90 * std), tf.float32)
    mask = tf.clip_by_value(high + 0.35 * low, 0.0, 1.0)
    mask = tf.nn.avg_pool2d(mask, ksize=7, strides=1, padding="SAME")
    return normalize_per_image(mask)


def intensity_map_tf(images):
    return normalize_per_image(rgb_to_gray01(images))

for sample_images, _ in train_ds.take(1):
    edge = combined_edge_map_tf(sample_images).numpy()
    tex = texture_map_tf(sample_images).numpy()
    morph = pseudo_morphology_mask_tf(sample_images).numpy()
    plt.figure(figsize=(12, 9))
    for i in range(min(3, sample_images.shape[0])):
        plt.subplot(3, 4, 4*i+1); plt.imshow(sample_images[i].numpy().astype("uint8")); plt.title("MRI"); plt.axis("off")
        plt.subplot(3, 4, 4*i+2); plt.imshow(edge[i,:,:,0], cmap="gray"); plt.title("Boundary prior"); plt.axis("off")
        plt.subplot(3, 4, 4*i+3); plt.imshow(tex[i,:,:,0], cmap="gray"); plt.title("Texture prior"); plt.axis("off")
        plt.subplot(3, 4, 4*i+4); plt.imshow(morph[i,:,:,0], cmap="gray"); plt.title("Pseudo morphology"); plt.axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# 6. RG-CBM model
# ============================================================
def concept_correlation_loss(concepts):
    concepts = tf.cast(concepts, tf.float32)
    k = tf.shape(concepts)[1]
    centered = concepts - tf.reduce_mean(concepts, axis=0, keepdims=True)
    batch = tf.cast(tf.shape(concepts)[0], tf.float32)
    cov = tf.matmul(centered, centered, transpose_a=True) / tf.maximum(batch - 1.0, 1.0)
    std = tf.sqrt(tf.linalg.diag_part(cov) + 1e-6)
    corr = cov / (tf.tensordot(std, std, axes=0) + 1e-6)
    off = corr * (1.0 - tf.eye(k, dtype=tf.float32))
    return tf.reduce_mean(tf.square(off))


def prototype_diversity_loss(prototypes):
    p = tf.nn.l2_normalize(prototypes, axis=-1)
    sim = tf.matmul(p, p, transpose_b=True)
    k = tf.shape(prototypes)[0]
    off = sim * (1.0 - tf.eye(k, dtype=tf.float32))
    return tf.reduce_mean(tf.square(off))


class RGCBMEfficientNetB1(tf.keras.Model):
    def __init__(
        self, num_classes, num_concepts=16, prototype_dim=128,
        n_boundary=4, n_texture=4, n_morphology=4, n_unguided=4,
        lambda_edge=0.35, lambda_texture=0.20, lambda_morph=0.15,
        lambda_diversity=0.03, lambda_disentangle=0.02,
        lambda_concept_sparse=5e-5, lambda_stability=0.05,
        classifier_l1=1e-5, concept_dropout=0.10,
        aug_rotation=0.025, aug_zoom=0.04, aug_translation=0.02, aug_contrast=0.06,
        backbone_weights="imagenet", input_shape=(240,240,3), use_augmentation=True,
    ):
        super().__init__(name="rg_cbm_effb1")
        self.num_classes = int(num_classes)
        self.num_concepts = int(num_concepts)
        self.prototype_dim = int(prototype_dim)
        self.n_boundary = int(n_boundary)
        self.n_texture = int(n_texture)
        self.n_morphology = int(n_morphology)
        self.n_unguided = int(n_unguided)

        self.lambda_edge = float(lambda_edge)
        self.lambda_texture = float(lambda_texture)
        self.lambda_morph = float(lambda_morph)
        self.lambda_diversity = float(lambda_diversity)
        self.lambda_disentangle = float(lambda_disentangle)
        self.lambda_concept_sparse = float(lambda_concept_sparse)
        self.lambda_stability = float(lambda_stability)
        self.use_augmentation = bool(use_augmentation)

        self.augmentation = tf.keras.Sequential([
            layers.RandomRotation(aug_rotation, fill_mode="nearest"),
            layers.RandomZoom(height_factor=aug_zoom, width_factor=aug_zoom, fill_mode="nearest"),
            layers.RandomTranslation(aug_translation, aug_translation, fill_mode="nearest"),
            layers.RandomContrast(aug_contrast),
        ], name="mri_safe_augmentation")

        self.backbone = tf.keras.applications.EfficientNetB1(
            include_top=False,
            weights=backbone_weights,
            input_shape=input_shape,
            pooling=None,
        )

        self.feature_projection = tf.keras.Sequential([
            layers.Conv2D(prototype_dim, 1, padding="same", use_bias=False),
            layers.BatchNormalization(),
            layers.Activation("swish"),
        ], name="feature_projection")

        self.concept_prototypes = self.add_weight(
            name="concept_prototypes",
            shape=(num_concepts, prototype_dim),
            initializer=tf.keras.initializers.RandomNormal(stddev=0.02),
            trainable=True,
        )

        # Auxiliary loss warm-up scale; does not change model structure.
        self.aux_scale = self.add_weight(
            name="auxiliary_loss_scale", shape=(),
            initializer=tf.keras.initializers.Constant(1.0), trainable=False,
        )

        self.concept_dropout = layers.Dropout(concept_dropout)
        self.classifier = layers.Dense(
            num_classes,
            activation=None,
            use_bias=True,
            kernel_regularizer=tf.keras.regularizers.L1(classifier_l1),
            name="sparse_linear_classifier",
        )
        self.softmax = layers.Softmax(dtype="float32")
        self.cls_loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
            from_logits=True, reduction="none"
        )

        self.total_loss_tracker = tf.keras.metrics.Mean(name="loss")
        self.cls_loss_tracker = tf.keras.metrics.Mean(name="classification_loss")
        self.edge_loss_tracker = tf.keras.metrics.Mean(name="edge_alignment_loss")
        self.texture_loss_tracker = tf.keras.metrics.Mean(name="texture_alignment_loss")
        self.morph_loss_tracker = tf.keras.metrics.Mean(name="morphology_alignment_loss")
        self.diversity_loss_tracker = tf.keras.metrics.Mean(name="prototype_diversity_loss")
        self.disentangle_loss_tracker = tf.keras.metrics.Mean(name="concept_disentanglement_loss")
        self.sparsity_loss_tracker = tf.keras.metrics.Mean(name="concept_sparsity_loss")
        self.stability_loss_tracker = tf.keras.metrics.Mean(name="concept_stability_loss")
        self.gradient_norm_tracker = tf.keras.metrics.Mean(name="gradient_norm")
        self.acc_metric = tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")

    @property
    def metrics(self):
        return [
            self.total_loss_tracker, self.cls_loss_tracker, self.edge_loss_tracker,
            self.texture_loss_tracker, self.morph_loss_tracker, self.diversity_loss_tracker,
            self.disentangle_loss_tracker, self.sparsity_loss_tracker,
            self.stability_loss_tracker, self.gradient_norm_tracker, self.acc_metric,
        ]

    def _concept_group_slices(self):
        b1 = self.n_boundary
        t1 = b1 + self.n_texture
        m1 = t1 + self.n_morphology
        u1 = m1 + self.n_unguided
        return slice(0, b1), slice(b1, t1), slice(t1, m1), slice(m1, u1)

    def compute_concept_maps(self, x, training=False):
        feature_map = self.backbone(x, training=training)
        projected = self.feature_projection(feature_map, training=training)
        f = tf.nn.l2_normalize(projected, axis=-1)
        p = tf.nn.l2_normalize(self.concept_prototypes, axis=-1)
        similarity = tf.einsum("bhwd,kd->bhwk", f, p)
        concept_maps = tf.sigmoid(4.0 * similarity)
        concepts = tf.reduce_mean(concept_maps, axis=[1, 2])
        return feature_map, projected, concept_maps, concepts

    def call(self, inputs, training=False):
        x = tf.clip_by_value(tf.cast(inputs, tf.float32), 0.0, 255.0)
        x_model = self.augmentation(x, training=True) if (training and self.use_augmentation) else x

        feature_map, projected, concept_maps, concepts = self.compute_concept_maps(x_model, training=training)
        clf_concepts = self.concept_dropout(concepts, training=True) if training else concepts
        logits = self.classifier(clf_concepts)
        probs = self.softmax(logits)

        # Priors are targets/metadata, not learnable branches.
        edge = tf.stop_gradient(combined_edge_map_tf(x_model))
        tex = tf.stop_gradient(texture_map_tf(x_model))
        morph = tf.stop_gradient(pseudo_morphology_mask_tf(x_model))
        intensity = tf.stop_gradient(intensity_map_tf(x_model))

        return {
            "logits": logits,
            "probs": probs,
            "concepts": concepts,
            "concept_maps": concept_maps,
            "edge_map": edge,
            "texture_map": tex,
            "morphology_map": morph,
            "intensity_map": intensity,
            "feature_map": feature_map,
            "model_input": x_model,
        }

    def compute_losses(self, x, y, outputs, sample_weight=None, training=False):
        logits = tf.cast(outputs["logits"], tf.float32)
        concepts = tf.cast(outputs["concepts"], tf.float32)
        concept_maps = tf.cast(outputs["concept_maps"], tf.float32)

        per_example_cls = self.cls_loss_fn(y, logits)
        if sample_weight is not None:
            sw = tf.cast(sample_weight, tf.float32)
            cls_loss = tf.reduce_sum(per_example_cls * sw) / (tf.reduce_sum(sw) + 1e-8)
        else:
            cls_loss = tf.reduce_mean(per_example_cls)

        feature_hw = tf.shape(concept_maps)[1:3]
        edge_resized = tf.image.resize(outputs["edge_map"], feature_hw)
        tex_resized = tf.image.resize(outputs["texture_map"], feature_hw)
        morph_resized = tf.image.resize(outputs["morphology_map"], feature_hw)

        b_slice, t_slice, m_slice, _ = self._concept_group_slices()
        boundary_map = tf.reduce_mean(concept_maps[..., b_slice], axis=-1, keepdims=True)
        texture_map = tf.reduce_mean(concept_maps[..., t_slice], axis=-1, keepdims=True)
        morphology_map = tf.reduce_mean(concept_maps[..., m_slice], axis=-1, keepdims=True)

        edge_loss = tf.reduce_mean(tf.square(boundary_map - edge_resized))
        texture_loss = tf.reduce_mean(tf.square(texture_map - tex_resized))
        morph_loss = tf.reduce_mean(tf.square(morphology_map - morph_resized))
        div_loss = prototype_diversity_loss(self.concept_prototypes)
        disent_loss = concept_correlation_loss(concepts)
        sparse_loss = tf.reduce_mean(tf.abs(concepts))

        # Stability branch: BOTH sides use the same inference-mode feature path.
        # This avoids penalizing differences caused only by BatchNorm training/inference mode.
        base_x = tf.cast(outputs["model_input"], tf.float32)
        if training:
            x_stab = tf.image.random_brightness(base_x, max_delta=5.0)
            x_stab = tf.image.random_contrast(x_stab, lower=0.98, upper=1.02)
        else:
            x_stab = tf.image.adjust_brightness(base_x, delta=3.0)
            x_stab = tf.image.adjust_contrast(x_stab, contrast_factor=1.02)
        x_stab = tf.clip_by_value(x_stab, 0.0, 255.0)
        _, _, _, concepts_stab_base = self.compute_concept_maps(base_x, training=False)
        _, _, _, concepts_stab_aug = self.compute_concept_maps(x_stab, training=False)
        stability_loss = tf.reduce_mean(tf.square(concepts_stab_base - concepts_stab_aug))

        reg_loss = tf.add_n(self.losses) if self.losses else tf.constant(0.0, tf.float32)
        auxiliary = (
            self.lambda_edge * edge_loss +
            self.lambda_texture * texture_loss +
            self.lambda_morph * morph_loss +
            self.lambda_diversity * div_loss +
            self.lambda_disentangle * disent_loss +
            self.lambda_concept_sparse * sparse_loss +
            self.lambda_stability * stability_loss
        )
        total = cls_loss + tf.cast(self.aux_scale, tf.float32) * auxiliary + reg_loss
        return total, cls_loss, edge_loss, texture_loss, morph_loss, div_loss, disent_loss, sparse_loss, stability_loss

    def train_step(self, data):
        x, y, sample_weight = tf.keras.utils.unpack_x_y_sample_weight(data)
        with tf.GradientTape() as tape:
            outputs = self(x, training=True)
            losses = self.compute_losses(x, y, outputs, sample_weight, training=True)

        # CRITICAL FIX: update every variable that is currently trainable.
        train_vars = self.trainable_variables
        grads = tape.gradient(losses[0], train_vars)
        grads_and_vars = [(g, v) for g, v in zip(grads, train_vars) if g is not None]
        if not grads_and_vars:
            raise RuntimeError("No gradients were produced. Check trainable variables and the loss graph.")
        grad_norm = tf.linalg.global_norm([g for g, _ in grads_and_vars])
        self.optimizer.apply_gradients(grads_and_vars)

        self._update_metrics(losses, y, outputs, sample_weight, grad_norm)
        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        x, y, sample_weight = tf.keras.utils.unpack_x_y_sample_weight(data)
        outputs = self(x, training=False)
        losses = self.compute_losses(x, y, outputs, sample_weight, training=False)
        self._update_metrics(losses, y, outputs, sample_weight, tf.constant(0.0, tf.float32))
        return {m.name: m.result() for m in self.metrics}

    def _update_metrics(self, losses, y, outputs, sample_weight, grad_norm):
        total, cls, edge, tex, morph, div, disent, sparse, stab = losses
        self.total_loss_tracker.update_state(total)
        self.cls_loss_tracker.update_state(cls)
        self.edge_loss_tracker.update_state(edge)
        self.texture_loss_tracker.update_state(tex)
        self.morph_loss_tracker.update_state(morph)
        self.diversity_loss_tracker.update_state(div)
        self.disentangle_loss_tracker.update_state(disent)
        self.sparsity_loss_tracker.update_state(sparse)
        self.stability_loss_tracker.update_state(stab)
        self.gradient_norm_tracker.update_state(grad_norm)
        self.acc_metric.update_state(y, outputs["logits"], sample_weight=sample_weight)

    def predict_from_concepts(self, concepts):
        logits = self.classifier(tf.cast(concepts, tf.float32))
        return logits, self.softmax(logits)


In [ ]:
# ============================================================
# 7. Build model
# ============================================================
model = RGCBMEfficientNetB1(
    num_classes=num_classes,
    num_concepts=NUM_CONCEPTS,
    prototype_dim=PROTOTYPE_DIM,
    n_boundary=NUM_BOUNDARY_CONCEPTS,
    n_texture=NUM_TEXTURE_CONCEPTS,
    n_morphology=NUM_MORPHOLOGY_CONCEPTS,
    n_unguided=NUM_UNGUIDED_CONCEPTS,
    lambda_edge=LAMBDA_EDGE,
    lambda_texture=LAMBDA_TEXTURE,
    lambda_morph=LAMBDA_MORPH,
    lambda_diversity=LAMBDA_DIVERSITY,
    lambda_disentangle=LAMBDA_DISENTANGLE,
    lambda_concept_sparse=LAMBDA_CONCEPT_SPARSE,
    lambda_stability=LAMBDA_STABILITY,
    classifier_l1=CLASSIFIER_L1,
    concept_dropout=CONCEPT_DROPOUT,
    aug_rotation=AUG_ROTATION,
    aug_zoom=AUG_ZOOM,
    aug_translation=AUG_TRANSLATION,
    aug_contrast=AUG_CONTRAST,
    input_shape=(*IMG_SIZE, 3),
    backbone_weights="imagenet",
    use_augmentation=True,
)

_ = model(tf.zeros((1, *IMG_SIZE, 3), dtype=tf.float32), training=False)


def set_backbone_frozen(model):
    model.backbone.trainable = False


def set_fine_tuning(model, last_n_layers=60):
    model.backbone.trainable = True
    for layer in model.backbone.layers:
        layer.trainable = False
    for layer in model.backbone.layers[-last_n_layers:]:
        # Keep BatchNorm frozen for small medical-imaging datasets.
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
        else:
            layer.trainable = True


def print_trainable_audit(model, title):
    print("\n" + title)
    print("Trainable tensors:", len(model.trainable_variables))
    print("Trainable parameters:", sum(int(np.prod(v.shape)) for v in model.trainable_variables))
    print("Classifier trainable:", model.classifier.trainable)
    print("Feature projection trainable:", model.feature_projection.trainable)
    print("Backbone trainable layers:", sum(int(l.trainable) for l in model.backbone.layers), "/", len(model.backbone.layers))
    for v in model.trainable_variables[:12]:
        print(" -", v.name, tuple(v.shape))


def make_optimizer(lr):
    try:
        return tf.keras.optimizers.AdamW(
            learning_rate=lr, weight_decay=1e-5, clipnorm=1.0
        )
    except Exception:
        return tf.keras.optimizers.Adam(learning_rate=lr, clipnorm=1.0)


class AuxiliaryCurriculum(tf.keras.callbacks.Callback):
    """Classification-first warm-up, then a smooth radiomics-guidance ramp."""
    def __init__(self, classification_only_epochs=3, ramp_epochs=5, target_scale=1.0):
        super().__init__()
        self.classification_only_epochs = max(0, int(classification_only_epochs))
        self.ramp_epochs = max(1, int(ramp_epochs))
        self.target_scale = float(target_scale)

    def on_epoch_begin(self, epoch, logs=None):
        if epoch < self.classification_only_epochs:
            scale = 0.0
        else:
            ramp_index = epoch - self.classification_only_epochs + 1
            scale = self.target_scale * min(1.0, ramp_index / float(self.ramp_epochs))
        self.model.aux_scale.assign(scale)
        print(f"Auxiliary-loss scale: {scale:.3f}")


class DriveEpochBackup(tf.keras.callbacks.Callback):
    def __init__(self, stage_name, tracked_files):
        super().__init__()
        self.stage_name = stage_name
        self.tracked_files = list(tracked_files)

    def on_epoch_end(self, epoch, logs=None):
        state = {
            "stage": self.stage_name,
            "epoch": int(epoch + 1),
            "timestamp": datetime.now().isoformat(),
            "metrics": {k: float(v) for k, v in (logs or {}).items()},
        }
        state_path = RUN_DIR / f"{self.stage_name}_latest_state.json"
        with open(state_path, "w") as f:
            json.dump(state, f, indent=2)
        sync_selected_to_drive(self.tracked_files + [state_path.name])


In [ ]:
# ============================================================
# 8. Stage 1: train concept head / projection / classifier
# ============================================================
HEAD_CKPT = RUN_DIR / "best_head.weights.h5"
HEAD_LOG = RUN_DIR / "training_head.csv"

set_backbone_frozen(model)
model.aux_scale.assign(0.0)
model.compile(optimizer=make_optimizer(HEAD_LR))
print_trainable_audit(model, "Stage 1 trainable audit")

# Store initial weights so we can prove that the classifier now learns.
initial_classifier_kernel = model.classifier.get_weights()[0].copy()
initial_prototypes = model.concept_prototypes.numpy().copy()

head_callbacks = [
    AuxiliaryCurriculum(
        AUX_CLASSIFICATION_ONLY_EPOCHS, AUX_RAMP_EPOCHS, AUX_TARGET_SCALE
    ),
    tf.keras.callbacks.TerminateOnNaN(),
    tf.keras.callbacks.ModelCheckpoint(
        str(HEAD_CKPT), monitor="val_accuracy", mode="max",
        save_best_only=True, save_weights_only=True, verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_classification_loss", mode="min", factor=0.5,
        patience=3, min_lr=2e-6, verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", mode="max", patience=6,
        restore_best_weights=True, verbose=1,
    ),
    tf.keras.callbacks.CSVLogger(str(HEAD_LOG), append=False),
    DriveEpochBackup("head", [HEAD_CKPT.name, HEAD_LOG.name]),
]

history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    class_weight=class_weight,
    callbacks=head_callbacks,
)

classifier_delta = float(np.linalg.norm(model.classifier.get_weights()[0] - initial_classifier_kernel))
prototype_delta = float(np.linalg.norm(model.concept_prototypes.numpy() - initial_prototypes))
print("Classifier kernel change (should be > 0):", classifier_delta)
print("Prototype change (should be > 0):", prototype_delta)

with open(RUN_DIR / "stage1_training_audit.json", "w") as f:
    json.dump({"classifier_kernel_l2_change": classifier_delta, "prototype_l2_change": prototype_delta}, f, indent=2)
sync_selected_to_drive(["stage1_training_audit.json", HEAD_CKPT.name, HEAD_LOG.name])


In [ ]:
# ============================================================
# 9. Stage 2: selective EfficientNetB1 fine-tuning
# ============================================================
FINETUNE_CKPT = RUN_DIR / "best_finetune.weights.h5"
FINETUNE_LOG = RUN_DIR / "training_finetune.csv"
BEST_CKPT = RUN_DIR / "best_rg_cbm_effb1.weights.h5"

# Start fine-tuning from the best head checkpoint.
if HEAD_CKPT.exists():
    model.load_weights(str(HEAD_CKPT))

set_fine_tuning(model, FINE_TUNE_LAST_N_LAYERS)
model.aux_scale.assign(AUX_TARGET_SCALE)
model.compile(optimizer=make_optimizer(FINETUNE_LR))
print_trainable_audit(model, "Stage 2 trainable audit")

finetune_callbacks = [
    tf.keras.callbacks.TerminateOnNaN(),
    tf.keras.callbacks.ModelCheckpoint(
        str(FINETUNE_CKPT), monitor="val_accuracy", mode="max",
        save_best_only=True, save_weights_only=True, verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_classification_loss", mode="min", factor=0.5,
        patience=3, min_lr=2e-7, verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", mode="max", patience=7,
        restore_best_weights=True, verbose=1,
    ),
    tf.keras.callbacks.CSVLogger(str(FINETUNE_LOG), append=False),
    DriveEpochBackup("finetune", [FINETUNE_CKPT.name, FINETUNE_LOG.name]),
]

history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINETUNE,
    class_weight=class_weight,
    callbacks=finetune_callbacks,
)

head_best = float(np.max(history_head.history.get("val_accuracy", [0.0])))
finetune_best = float(np.max(history_finetune.history.get("val_accuracy", [0.0])))
chosen = FINETUNE_CKPT if finetune_best >= head_best else HEAD_CKPT
print(f"Best head val accuracy: {head_best:.4f}")
print(f"Best fine-tune val accuracy: {finetune_best:.4f}")
print("Chosen checkpoint:", chosen)

model.load_weights(str(chosen))
shutil.copy2(chosen, BEST_CKPT)


def history_to_frame(h, stage):
    df = pd.DataFrame(h.history)
    df["stage"] = stage
    df["epoch_in_stage"] = np.arange(1, len(df) + 1)
    return df

history_df = pd.concat([
    history_to_frame(history_head, "head"),
    history_to_frame(history_finetune, "fine_tune"),
], ignore_index=True)
history_df.to_csv(RUN_DIR / "history_combined.csv", index=False)

for metric in ["accuracy", "loss", "classification_loss", "edge_alignment_loss", "texture_alignment_loss", "morphology_alignment_loss", "gradient_norm"]:
    plt.figure(figsize=(8, 5))
    if metric in history_df:
        plt.plot(history_df[metric].values, marker="o", label=metric)
    val_metric = "val_" + metric
    if val_metric in history_df:
        plt.plot(history_df[val_metric].values, marker="o", label=val_metric)
    plt.title(metric)
    plt.xlabel("Epoch")
    plt.ylabel(metric)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(RUN_DIR / f"{metric}.png", dpi=160)
    plt.show()

sync_to = [BEST_CKPT.name, "history_combined.csv", HEAD_LOG.name, FINETUNE_LOG.name]
sync_selected_to_drive(sync_to)


In [ ]:
# ============================================================
# 10. Classification evaluation
# ============================================================
if BEST_CKPT.exists():
    model.load_weights(str(BEST_CKPT))

# test_step is deterministic and cannot update BatchNorm state.
test_metrics = model.evaluate(test_ds, return_dict=True, verbose=1)
print("Keras test metrics:", json.dumps({k: float(v) for k, v in test_metrics.items()}, indent=2))


def predict_dataset(model, ds):
    y_true, y_pred, probs_all, concepts_all = [], [], [], []
    for x, y in ds:
        out = model(x, training=False)
        probs = out["probs"].numpy()
        concepts = out["concepts"].numpy()
        y_true.extend(y.numpy().tolist())
        y_pred.extend(np.argmax(probs, axis=1).tolist())
        probs_all.append(probs)
        concepts_all.append(concepts)
    return {
        "y_true": np.asarray(y_true),
        "y_pred": np.asarray(y_pred),
        "probs": np.concatenate(probs_all),
        "concepts": np.concatenate(concepts_all),
    }


test_outputs = predict_dataset(model, test_ds)
sk_acc = accuracy_score(test_outputs["y_true"], test_outputs["y_pred"])
print(f"Independent prediction accuracy: {sk_acc:.6f}")
print(f"Keras-vs-independent difference: {abs(float(test_metrics['accuracy']) - sk_acc):.8f}")

report_text = classification_report(
    test_outputs["y_true"], test_outputs["y_pred"],
    target_names=class_names, digits=4,
)
print(report_text)
with open(RUN_DIR / "classification_report.txt", "w") as f:
    f.write(report_text)

cm = confusion_matrix(test_outputs["y_true"], test_outputs["y_pred"])
pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(RUN_DIR / "confusion_matrix.csv")

plt.figure(figsize=(7, 6))
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xticks(np.arange(num_classes), class_names, rotation=45, ha="right")
plt.yticks(np.arange(num_classes), class_names)
plt.colorbar()
for i in range(num_classes):
    for j in range(num_classes):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")
plt.tight_layout()
plt.savefig(RUN_DIR / "confusion_matrix.png", dpi=180, bbox_inches="tight")
plt.show()

with open(RUN_DIR / "test_metrics.json", "w") as f:
    json.dump({k: float(v) for k, v in test_metrics.items()} | {"independent_accuracy": float(sk_acc)}, f, indent=2)

sync_selected_to_drive(["classification_report.txt", "confusion_matrix.csv", "confusion_matrix.png", "test_metrics.json"])


## 11. Automatic concept grounding and naming

Grounding is computed **per image** and then averaged.


In [ ]:
# ============================================================
# 11. Automatic concept grounding and naming
# ============================================================
def safe_corr(a, b):
    a = np.asarray(a).ravel()
    b = np.asarray(b).ravel()
    if a.std() < 1e-8 or b.std() < 1e-8:
        return 0.0
    return float(np.corrcoef(a, b)[0, 1])


def descriptor_maps_for_batch_np(images_np, map_size):
    images_tf = tf.convert_to_tensor(images_np, dtype=tf.float32)
    desc = {
        "edge": combined_edge_map_tf(images_tf).numpy(),
        "texture": texture_map_tf(images_tf).numpy(),
        "morphology": pseudo_morphology_mask_tf(images_tf).numpy(),
        "intensity": intensity_map_tf(images_tf).numpy(),
    }
    return {k: tf.image.resize(v, map_size).numpy()[..., 0] for k, v in desc.items()}


def training_group_for_index(model, k):
    b, t, m, u = model._concept_group_slices()
    if b.start <= k < b.stop:
        return "boundary"
    if t.start <= k < t.stop:
        return "texture"
    if m.start <= k < m.stop:
        return "morphology"
    return "unguided"


def choose_concept_name(best_descriptor, corr_value, concept_idx):
    names = {
        "edge": ["irregular boundary pattern", "sharp edge-density pattern", "boundary transition pattern", "margin irregularity pattern"],
        "texture": ["heterogeneous texture pattern", "coarse texture variation", "local texture-complexity pattern", "granular intensity-variation pattern"],
        "morphology": ["compact lesion-region pattern", "diffuse morphology pattern", "asymmetric region pattern", "lesion-shape pattern"],
        "intensity": ["hyperintense tissue pattern", "hypointense tissue pattern", "intensity-variation pattern", "contrast-related intensity pattern"],
    }
    base = names[best_descriptor][concept_idx % 4]
    if best_descriptor == "intensity" and corr_value < 0:
        base = "hypointense tissue pattern"
    return base


def generate_automatic_concept_names(model, ds, n_batches=16, weak_threshold=0.08):
    corrs = {d: [[] for _ in range(model.num_concepts)] for d in ["edge", "texture", "morphology", "intensity"]}

    for batch_idx, (x, _) in enumerate(ds):
        if batch_idx >= n_batches:
            break
        out = model(x, training=False)
        cmaps = out["concept_maps"].numpy()
        h, w = cmaps.shape[1], cmaps.shape[2]
        desc = descriptor_maps_for_batch_np(x.numpy(), (h, w))

        for k in range(model.num_concepts):
            for i in range(cmaps.shape[0]):
                for d in desc:
                    corrs[d][k].append(safe_corr(cmaps[i, :, :, k], desc[d][i]))

    rows, names = [], []
    for k in range(model.num_concepts):
        mean_corrs = {d: float(np.nanmean(corrs[d][k])) for d in corrs}
        best_desc = max(mean_corrs, key=lambda d: abs(mean_corrs[d]))
        best_corr = mean_corrs[best_desc]
        group = training_group_for_index(model, k)
        readable_base = choose_concept_name(best_desc, best_corr, k)
        readable = f"{readable_base} [{group}, auto-{k+1}]"
        names.append(readable)
        row = {
            "concept_index": k,
            "concept_name": readable,
            "training_group": group,
            "best_grounding_descriptor": best_desc,
            "best_grounding_correlation": best_corr,
            "grounding_strength": abs(best_corr),
            "weak_grounding": bool(abs(best_corr) < weak_threshold),
        }
        row.update({f"corr_{d}": mean_corrs[d] for d in mean_corrs})
        rows.append(row)
    return names, pd.DataFrame(rows)


concept_names, grounding_df = generate_automatic_concept_names(model, val_ds, n_batches=16)
grounding_df.to_csv(RUN_DIR / "automatic_concept_grounding_and_names.csv", index=False)

for i, n in enumerate(concept_names, 1):
    print(f"{i:02d}. {n}")
display(grounding_df)

plt.figure(figsize=(12, 5))
plt.bar(np.arange(NUM_CONCEPTS), grounding_df["grounding_strength"].values)
plt.xticks(np.arange(NUM_CONCEPTS), [str(i+1) for i in range(NUM_CONCEPTS)])
plt.xlabel("Concept")
plt.ylabel("|mean per-image grounding correlation|")
plt.title("Automatic concept grounding strength")
plt.tight_layout()
plt.savefig(RUN_DIR / "concept_grounding_strength.png", dpi=180)
plt.show()

concept_df = pd.DataFrame(test_outputs["concepts"], columns=concept_names)
concept_df["true_label"] = [class_names[i] for i in test_outputs["y_true"]]
concept_df["predicted_label"] = [class_names[i] for i in test_outputs["y_pred"]]
concept_df["confidence"] = np.max(test_outputs["probs"], axis=1)
concept_df.to_csv(RUN_DIR / "test_concept_report.csv", index=False)

sync_selected_to_drive([
    "automatic_concept_grounding_and_names.csv",
    "concept_grounding_strength.png",
    "test_concept_report.csv",
])


## 12. Exact linear concept evidence
These values are the evidence passed to the LLM.


In [ ]:
# ============================================================
# 12. Exact contribution decomposition + single-image explanation
# ============================================================
def list_image_files(folder):
    return sorted([p for p in Path(folder).rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS])


def load_image_array(image_path, image_size=IMG_SIZE):
    pil = tf.keras.utils.load_img(image_path, color_mode="rgb", target_size=image_size)
    arr = tf.keras.utils.img_to_array(pil).astype("float32")
    return arr[None, ...], pil


def resize_map_to_image(map_hw, image_size=IMG_SIZE):
    m = map_hw[..., None] if map_hw.ndim == 2 else map_hw
    m = tf.image.resize(m[None, ...], image_size).numpy()[0, ..., 0]
    return (m - m.min()) / (m.max() - m.min() + 1e-8)


def exact_linear_evidence(model, concepts, probs, concept_names, grounding_df):
    W, b = model.classifier.get_weights()
    concepts = np.asarray(concepts, dtype=np.float64)
    probs = np.asarray(probs, dtype=np.float64)

    order = np.argsort(-probs)
    pred_idx, runner_idx = int(order[0]), int(order[1])
    pred_contrib = concepts * W[:, pred_idx]
    runner_contrib = concepts * W[:, runner_idx]
    margin_contrib = concepts * (W[:, pred_idx] - W[:, runner_idx])

    pred_logit_recon = float(b[pred_idx] + pred_contrib.sum())
    runner_logit_recon = float(b[runner_idx] + runner_contrib.sum())
    margin_recon = float((b[pred_idx] - b[runner_idx]) + margin_contrib.sum())

    rows = []
    for k in range(len(concepts)):
        g = grounding_df.iloc[k]
        rows.append({
            "concept_index": int(k),
            "concept_name": concept_names[k],
            "training_group": str(g["training_group"]),
            "grounding_descriptor": str(g["best_grounding_descriptor"]),
            "grounding_correlation": float(g["best_grounding_correlation"]),
            "grounding_strength": float(g["grounding_strength"]),
            "weak_grounding": bool(g["weak_grounding"]),
            "activation": float(concepts[k]),
            "predicted_class_weight": float(W[k, pred_idx]),
            "predicted_class_contribution": float(pred_contrib[k]),
            "runner_up_weight": float(W[k, runner_idx]),
            "runner_up_contribution": float(runner_contrib[k]),
            "margin_contribution": float(margin_contrib[k]),
        })

    return {
        "predicted_index": pred_idx,
        "predicted_class": class_names[pred_idx],
        "predicted_probability": float(probs[pred_idx]),
        "runner_up_index": runner_idx,
        "runner_up_class": class_names[runner_idx],
        "runner_up_probability": float(probs[runner_idx]),
        "predicted_bias": float(b[pred_idx]),
        "runner_up_bias": float(b[runner_idx]),
        "predicted_logit_reconstructed": pred_logit_recon,
        "runner_up_logit_reconstructed": runner_logit_recon,
        "predicted_vs_runner_up_margin_reconstructed": margin_recon,
        "concepts": rows,
    }


def build_llm_evidence_packet(exact_ev, top_n=5):
    rows = exact_ev["concepts"]
    support = sorted([r for r in rows if r["margin_contribution"] > 0], key=lambda r: r["margin_contribution"], reverse=True)[:top_n]
    oppose = sorted([r for r in rows if r["margin_contribution"] < 0], key=lambda r: r["margin_contribution"])[:top_n]

    total_positive = sum(max(0.0, r["margin_contribution"]) for r in rows) + 1e-12
    total_negative = sum(abs(min(0.0, r["margin_contribution"])) for r in rows) + 1e-12
    for r in support:
        r["positive_margin_share"] = float(r["margin_contribution"] / total_positive)
    for r in oppose:
        r["negative_margin_share"] = float(abs(r["margin_contribution"]) / total_negative)

    return {
        "prediction": {
            "class": exact_ev["predicted_class"],
            "probability": exact_ev["predicted_probability"],
            "runner_up_class": exact_ev["runner_up_class"],
            "runner_up_probability": exact_ev["runner_up_probability"],
        },
        "linear_decomposition": {
            "predicted_bias": exact_ev["predicted_bias"],
            "predicted_logit_reconstructed": exact_ev["predicted_logit_reconstructed"],
            "runner_up_bias": exact_ev["runner_up_bias"],
            "runner_up_logit_reconstructed": exact_ev["runner_up_logit_reconstructed"],
            "predicted_vs_runner_up_margin": exact_ev["predicted_vs_runner_up_margin_reconstructed"],
        },
        "main_supporting_margin_concepts": support,
        "main_opposing_margin_concepts": oppose,
        "evidence_scope": (
            "All findings are learned RG-CBM concepts automatically grounded against image-derived "
            "boundary, texture, morphology, or intensity descriptors. They are not manual clinical annotations."
        ),
    }


def explain_image(model, image_path, top_n=5, save_path=None, show_plot=True):
    x, pil = load_image_array(image_path)
    out = model(tf.constant(x), training=False)
    probs = out["probs"].numpy()[0]
    concepts = out["concepts"].numpy()[0]
    cmaps = out["concept_maps"].numpy()[0]

    exact_ev = exact_linear_evidence(model, concepts, probs, concept_names, grounding_df)
    packet = build_llm_evidence_packet(exact_ev, top_n=top_n)

    # Rank by absolute predicted-vs-runner-up margin contribution for visualization.
    ranked = sorted(exact_ev["concepts"], key=lambda r: abs(r["margin_contribution"]), reverse=True)[:top_n]

    if show_plot or save_path is not None:
        ncols = top_n + 1
        plt.figure(figsize=(3.2*ncols, 4.0))
        plt.subplot(1, ncols, 1)
        plt.imshow(pil)
        plt.title(f"{exact_ev['predicted_class']}\n{exact_ev['predicted_probability']:.1%}")
        plt.axis("off")

        for j, row in enumerate(ranked, 2):
            cmap = resize_map_to_image(cmaps[:, :, row["concept_index"]])
            plt.subplot(1, ncols, j)
            plt.imshow(pil)
            plt.imshow(cmap, alpha=0.45, cmap="jet")
            short = re.sub(r"\s*\[.*?\]\s*", "", row["concept_name"])
            plt.title(f"{short}\nmargin={row['margin_contribution']:+.3f}", fontsize=9)
            plt.axis("off")
        plt.tight_layout()
        if save_path is not None:
            plt.savefig(save_path, dpi=180, bbox_inches="tight")
        if show_plot:
            plt.show()
        else:
            plt.close()

    return {
        "image_path": str(image_path),
        "predicted_class": exact_ev["predicted_class"],
        "confidence": exact_ev["predicted_probability"],
        "runner_up_class": exact_ev["runner_up_class"],
        "runner_up_probability": exact_ev["runner_up_probability"],
        "exact_evidence": exact_ev,
        "llm_evidence_packet": packet,
        "ranked_concepts": ranked,
    }


all_test_images = list_image_files(TEST_DIR)
random.Random(SEED).shuffle(all_test_images)
sample_path = all_test_images[0]
SINGLE_FIG = RUN_DIR / "single_case_concept_explanation.png"
single_exp = explain_image(model, sample_path, top_n=5, save_path=SINGLE_FIG, show_plot=True)
print(json.dumps(single_exp["llm_evidence_packet"], indent=2))
sync_selected_to_drive([SINGLE_FIG.name])


## 13. LLM doctor-understandable report 

The LLM is still a **communication layer only**. It receives the structured RG-CBM evidence packet after the predicted class has already been produced.

This notebook supports two free routes:

1. **Gemini 2.5 Flash (`LLM_BACKEND="gemini"`)** — easiest in Colab because it does not consume the TensorFlow GPU memory. Obtain a Gemini API key from Google AI Studio and store it as the Colab secret `GEMINI_API_KEY` (or enter it when prompted). Google's free tier has rate limits and can change over time.
2. **Local Qwen (`LLM_BACKEND="local_qwen"`)** — uses `Qwen/Qwen2.5-1.5B-Instruct` through Hugging Face Transformers. No API key and no per-request charge. It defaults to CPU to avoid competing with TensorFlow for GPU memory; set `LOCAL_LLM_DEVICE="cuda"` only if your runtime has enough free GPU memory.

If Gemini is selected but fails because of quota/key/network issues, `ALLOW_LOCAL_FALLBACK=True` automatically tries the local Qwen backend.

Every report is saved together with the exact evidence JSON supplied to the LLM for reproducibility.


In [ ]:
# ============================================================
# 13. LLM doctor-understandable report from structured RG-CBM evidence
#     Backends: Gemini free tier OR fully local Qwen
# ============================================================
LLM_INSTRUCTIONS = """
You are a clinical-facing explanation writer for a research-only brain MRI classifier.
You are NOT making a diagnosis and you must NOT add medical findings that are absent from the supplied model evidence.

Use only the JSON evidence provided by the RG-CBM. The evidence consists of an AI-predicted class,
model confidence, the runner-up class, learned concept activations, exact linear classifier weights/contributions,
predicted-vs-runner-up margin contributions, and automatic grounding correlations.

Rules:
1. Explain why the MODEL selected its predicted class; do not claim the patient truly has that tumor.
2. Do not infer tumor location, size, edema, enhancement pattern, MRI sequence, demographics, symptoms, grade,
   pathology, treatment, prognosis, or any other fact that is not explicitly in the JSON.
3. Concept names are automatically grounded model concepts, not expert annotations. If grounding_strength is low
   or weak_grounding is true, describe that concept cautiously.
4. Give greatest emphasis to positive predicted-vs-runner-up margin contributions because they explain why the
   predicted class beat the closest alternative. Mention meaningful opposing evidence when present.
5. Do not convert confidence into certainty. Do not provide management or treatment recommendations.
6. Use concise, doctor-understandable language while keeping the reasoning traceable to the supplied numbers.

Return Markdown with these headings:
# Doctor-Understandable AI Explanation
## AI result
## Why the model favored this class
## Main supporting concept evidence
## Evidence against / uncertainty
## Why this class over the closest alternative
## Interpretation limits

For each main finding, include the concept name, its evidence type/grounding descriptor, activation,
margin contribution, and whether the grounding is strong/moderate/weak based on the supplied correlation.
End by stating that the report explains model reasoning and is not a clinical diagnosis.
""".strip()


def _secret_from_runtime(secret_name):
    value = os.getenv(secret_name)
    if value:
        return value
    if IN_COLAB:
        try:
            from google.colab import userdata
            value = userdata.get(secret_name)
            if value:
                os.environ[secret_name] = value
                return value
        except Exception:
            pass
    if IN_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            value = UserSecretsClient().get_secret(secret_name)
            if value:
                os.environ[secret_name] = value
                return value
        except Exception:
            pass
    return None


def get_gemini_api_key():
    key = _secret_from_runtime("GEMINI_API_KEY")
    if key:
        return key
    key = getpass("Enter GEMINI_API_KEY (input hidden; not saved): ").strip()
    if not key:
        raise RuntimeError("No GEMINI_API_KEY supplied.")
    os.environ["GEMINI_API_KEY"] = key
    return key


_gemini_client = None
_local_llm_model = None
_local_llm_tokenizer = None


def generate_report_gemini(evidence_packet, model_name=GEMINI_MODEL):
    """Cloud option. Designed to work with Google's Gemini API free tier."""
    global _gemini_client
    from google import genai
    from google.genai import types

    if _gemini_client is None:
        _gemini_client = genai.Client(api_key=get_gemini_api_key())

    payload = json.dumps(evidence_packet, indent=2)
    response = _gemini_client.models.generate_content(
        model=model_name,
        contents=(
            "Generate the doctor-understandable RG-CBM explanation from the following "
            "structured evidence only. Do not use outside clinical assumptions.\n\n" + payload
        ),
        config=types.GenerateContentConfig(
            system_instruction=LLM_INSTRUCTIONS,
            temperature=0.1,
            max_output_tokens=1400,
        ),
    )
    text = (response.text or "").strip()
    if not text:
        raise RuntimeError("Gemini returned an empty report.")
    return text


def _load_local_qwen():
    """Lazy-load Qwen only if local generation is actually requested."""
    global _local_llm_model, _local_llm_tokenizer
    if _local_llm_model is not None:
        return _local_llm_tokenizer, _local_llm_model

    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM

    print("Loading local LLM:", LOCAL_LLM_MODEL)
    _local_llm_tokenizer = AutoTokenizer.from_pretrained(LOCAL_LLM_MODEL)

    if LOCAL_LLM_DEVICE.lower() == "cuda" and torch.cuda.is_available():
        _local_llm_model = AutoModelForCausalLM.from_pretrained(
            LOCAL_LLM_MODEL,
            torch_dtype=torch.float16,
            device_map="cuda",
            low_cpu_mem_usage=True,
        )
    else:
        # CPU is slower, but it avoids TensorFlow/Qwen GPU-memory competition.
        _local_llm_model = AutoModelForCausalLM.from_pretrained(
            LOCAL_LLM_MODEL,
            torch_dtype=torch.float32,
            low_cpu_mem_usage=True,
        )
        _local_llm_model.to("cpu")

    _local_llm_model.eval()
    return _local_llm_tokenizer, _local_llm_model


def generate_report_local_qwen(evidence_packet):
    import torch
    tokenizer, llm = _load_local_qwen()
    payload = json.dumps(evidence_packet, indent=2)
    messages = [
        {"role": "system", "content": LLM_INSTRUCTIONS},
        {"role": "user", "content": (
            "Generate the doctor-understandable RG-CBM explanation using only this JSON evidence:\n\n" + payload
        )},
    ]
    encoded = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    device = next(llm.parameters()).device
    encoded = {k: v.to(device) for k, v in encoded.items()}
    input_len = encoded["input_ids"].shape[-1]

    with torch.inference_mode():
        output = llm.generate(
            **encoded,
            max_new_tokens=900,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(output[0][input_len:], skip_special_tokens=True).strip()
    if not text:
        raise RuntimeError("Local Qwen returned an empty report.")
    return text


def generate_llm_doctor_report(evidence_packet, max_retries=2):
    """Backend dispatcher with optional Gemini -> local-Qwen fallback."""
    if not USE_LLM_REPORTS or LLM_BACKEND == "none":
        raise RuntimeError("LLM report generation is disabled.")

    backend = LLM_BACKEND
    last_exc = None
    for attempt in range(max_retries):
        try:
            if backend == "gemini":
                return generate_report_gemini(evidence_packet)
            if backend == "local_qwen":
                return generate_report_local_qwen(evidence_packet)
            raise ValueError(f"Unknown LLM_BACKEND={backend!r}. Use 'gemini', 'local_qwen', or 'none'.")
        except Exception as exc:
            last_exc = exc
            print(f"LLM backend {backend!r} attempt {attempt+1} failed:", repr(exc))
            if attempt + 1 < max_retries:
                time.sleep(2 ** attempt)

    if backend == "gemini" and ALLOW_LOCAL_FALLBACK:
        print("Gemini unavailable; falling back to fully local Qwen.")
        return generate_report_local_qwen(evidence_packet)

    raise RuntimeError(f"LLM report generation failed using {backend}: {last_exc}")


def save_llm_report(exp, case_id="single_case"):
    report_dir = RUN_DIR / "llm_reports"
    report_dir.mkdir(parents=True, exist_ok=True)

    evidence_path = report_dir / f"{case_id}_evidence.json"
    report_path = report_dir / f"{case_id}_doctor_report.md"
    backend_path = report_dir / f"{case_id}_llm_backend.json"

    with open(evidence_path, "w", encoding="utf-8") as f:
        json.dump(exp["llm_evidence_packet"], f, indent=2)

    report = generate_llm_doctor_report(exp["llm_evidence_packet"])
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(report)
    with open(backend_path, "w", encoding="utf-8") as f:
        json.dump({
            "requested_backend": LLM_BACKEND,
            "gemini_model": GEMINI_MODEL if LLM_BACKEND == "gemini" else None,
            "local_model": LOCAL_LLM_MODEL if (LLM_BACKEND == "local_qwen" or ALLOW_LOCAL_FALLBACK) else None,
            "role": "Explanation verbalization only; no effect on classification.",
        }, f, indent=2)

    sync_selected_to_drive([
        str(evidence_path.relative_to(RUN_DIR)),
        str(report_path.relative_to(RUN_DIR)),
        str(backend_path.relative_to(RUN_DIR)),
    ])
    return report, report_path, evidence_path


if USE_LLM_REPORTS and LLM_BACKEND != "none":
    doctor_report, doctor_report_path, doctor_evidence_path = save_llm_report(single_exp, "single_case")
    display(Markdown(doctor_report))
    print("Saved LLM report:", doctor_report_path)
else:
    print("LLM disabled; structured evidence was generated and can be exported without an LLM call.")


In [ ]:
# ============================================================
# 14. Concept intervention
# ============================================================
def intervene_on_concepts(model, image_path, changes):
    x, _ = load_image_array(image_path)
    out = model(tf.constant(x), training=False)
    original_concepts = out["concepts"].numpy()[0]
    edited_concepts = original_concepts.copy()
    matched = []

    for key, value in changes.items():
        if isinstance(key, int):
            idx = int(key)
        else:
            candidates = [i for i, name in enumerate(concept_names) if str(key).lower() in name.lower()]
            if not candidates:
                print("No concept matched:", key)
                continue
            idx = candidates[0]
        edited_concepts[idx] = float(np.clip(value, 0.0, 1.0))
        matched.append((idx, concept_names[idx], original_concepts[idx], edited_concepts[idx]))

    _, original_probs = model.predict_from_concepts(original_concepts[None, :])
    _, edited_probs = model.predict_from_concepts(edited_concepts[None, :])
    original_probs = original_probs.numpy()[0]
    edited_probs = edited_probs.numpy()[0]

    table = pd.DataFrame({
        "class": class_names,
        "original_probability": original_probs,
        "edited_probability": edited_probs,
        "change": edited_probs - original_probs,
    }).sort_values("edited_probability", ascending=False)

    for idx, name, old, new in matched:
        print(f"{name}: {old:.3f} -> {new:.3f}")
    display(table)
    return table

# Example: ablate the strongest positive predicted-vs-runner-up concept.
supporting = single_exp["llm_evidence_packet"]["main_supporting_margin_concepts"]
if supporting:
    top_idx = supporting[0]["concept_index"]
    _ = intervene_on_concepts(model, sample_path, {top_idx: 0.0})


In [ ]:
# ============================================================
# 15. Faithfulness, stability, and grounding plausibility
# ============================================================
def faithfulness_top_concept_drop(model, ds, n_batches=12, mode="positive_support"):
    """
    mode='positive_support': remove the largest positive contribution to the predicted class.
    mode='absolute': reproduces the older absolute-contribution protocol for comparability.
    """
    rows, drops = [], []
    W, b = model.classifier.get_weights()

    for batch_idx, (x, _) in enumerate(ds):
        if batch_idx >= n_batches:
            break
        out = model(x, training=False)
        probs = out["probs"].numpy()
        concepts = out["concepts"].numpy()

        for i in range(concepts.shape[0]):
            pred = int(np.argmax(probs[i]))
            contrib = concepts[i] * W[:, pred]
            if mode == "positive_support" and np.any(contrib > 0):
                top_idx = int(np.argmax(np.where(contrib > 0, contrib, -np.inf)))
            else:
                top_idx = int(np.argmax(np.abs(contrib)))

            edited = concepts[i].copy()
            edited[top_idx] = 0.0
            _, edited_probs = model.predict_from_concepts(edited[None, :])
            edited_probs = edited_probs.numpy()[0]
            drop = float(probs[i, pred] - edited_probs[pred])
            drops.append(drop)
            rows.append({
                "predicted_class": class_names[pred],
                "removed_concept": concept_names[top_idx],
                "removed_contribution": float(contrib[top_idx]),
                "original_prob": float(probs[i, pred]),
                "edited_prob": float(edited_probs[pred]),
                "probability_drop": drop,
                "protocol": mode,
            })

    return pd.DataFrame(rows), float(np.mean(drops)), float(np.std(drops))


faith_pos_df, faith_pos_mean, faith_pos_std = faithfulness_top_concept_drop(model, test_ds, mode="positive_support")
faith_abs_df, faith_abs_mean, faith_abs_std = faithfulness_top_concept_drop(model, test_ds, mode="absolute")
faith_pos_df.to_csv(RUN_DIR / "faithfulness_positive_support_drop.csv", index=False)
faith_abs_df.to_csv(RUN_DIR / "faithfulness_absolute_protocol.csv", index=False)

print(f"Positive-support faithfulness drop: {faith_pos_mean:.4f} ± {faith_pos_std:.4f}")
print(f"Absolute-contribution protocol:     {faith_abs_mean:.4f} ± {faith_abs_std:.4f}")


def stability_under_augmentation(model, ds, n_batches=12):
    maes, cosines = [], []
    for batch_idx, (x, _) in enumerate(ds):
        if batch_idx >= n_batches:
            break
        # Fixed mild perturbation for reproducible evaluation.
        x_aug = tf.image.adjust_brightness(x, delta=4.0)
        x_aug = tf.image.adjust_contrast(x_aug, contrast_factor=1.03)
        x_aug = tf.clip_by_value(x_aug, 0.0, 255.0)
        c1 = model(x, training=False)["concepts"].numpy()
        c2 = model(x_aug, training=False)["concepts"].numpy()
        maes.extend(np.mean(np.abs(c1 - c2), axis=1).tolist())
        cos = np.sum(c1*c2, axis=1) / (np.linalg.norm(c1, axis=1) * np.linalg.norm(c2, axis=1) + 1e-8)
        cosines.extend(cos.tolist())
    return {
        "mean_concept_mae": float(np.mean(maes)),
        "std_concept_mae": float(np.std(maes)),
        "mean_concept_cosine": float(np.mean(cosines)),
        "std_concept_cosine": float(np.std(cosines)),
    }


stability_metrics = stability_under_augmentation(model, test_ds)
with open(RUN_DIR / "stability_metrics.json", "w") as f:
    json.dump(stability_metrics, f, indent=2)
print(json.dumps(stability_metrics, indent=2))

plausibility_df = grounding_df.copy()
plausibility_df["clinical_plausibility_proxy"] = plausibility_df["grounding_strength"]
plausibility_df.to_csv(RUN_DIR / "clinical_plausibility_proxy_scores.csv", index=False)
print("Mean grounding/plausibility proxy:", float(plausibility_df["clinical_plausibility_proxy"].mean()))

with open(RUN_DIR / "explanation_metrics_summary.json", "w") as f:
    json.dump({
        "faithfulness_positive_support_mean": faith_pos_mean,
        "faithfulness_positive_support_std": faith_pos_std,
        "faithfulness_absolute_protocol_mean": faith_abs_mean,
        "faithfulness_absolute_protocol_std": faith_abs_std,
        **stability_metrics,
        "mean_grounding_strength": float(plausibility_df["grounding_strength"].mean()),
    }, f, indent=2)

sync_selected_to_drive([
    "faithfulness_positive_support_drop.csv",
    "faithfulness_absolute_protocol.csv",
    "stability_metrics.json",
    "clinical_plausibility_proxy_scores.csv",
    "explanation_metrics_summary.json",
])


In [ ]:
# ============================================================
# 16. Prototype patches for visual concept evidence
# ============================================================
def crop_patch_around_location(image, center_yx, patch_size=80):
    h, w = image.shape[:2]
    cy, cx = center_yx
    half = patch_size // 2
    y1, y2 = max(0, cy-half), min(h, cy+half)
    x1, x2 = max(0, cx-half), min(w, cx+half)
    return image[y1:y2, x1:x2]


def collect_top_prototype_patches(model, ds, top_per_concept=5, n_batches=20):
    storage = {k: [] for k in range(model.num_concepts)}
    for batch_idx, (x, y) in enumerate(ds):
        if batch_idx >= n_batches:
            break
        out = model(x, training=False)
        cmaps = out["concept_maps"].numpy()
        x_np = x.numpy().astype(np.uint8)
        B, h, w, K = cmaps.shape
        y_np = y.numpy()

        for i in range(B):
            for k in range(K):
                cmap = cmaps[i, :, :, k]
                pos = np.unravel_index(np.argmax(cmap), cmap.shape)
                score = float(cmap[pos])
                cy = int((pos[0] + 0.5) * IMG_SIZE[0] / h)
                cx = int((pos[1] + 0.5) * IMG_SIZE[1] / w)
                patch = crop_patch_around_location(x_np[i], (cy, cx), patch_size=80)
                storage[k].append((score, patch, class_names[int(y_np[i])]))

    return {k: sorted(storage[k], key=lambda t: t[0], reverse=True)[:top_per_concept] for k in storage}


prototype_patches = collect_top_prototype_patches(model, train_ds, top_per_concept=5, n_batches=20)


def plot_concept_prototypes(concept_idx):
    items = prototype_patches[concept_idx]
    n = len(items)
    plt.figure(figsize=(3*n, 3.8))
    for j, (score, patch, label_name) in enumerate(items):
        plt.subplot(1, n, j+1)
        plt.imshow(patch)
        plt.title(f"{label_name}\nscore={score:.2f}", fontsize=9)
        plt.axis("off")
    plt.suptitle(concept_names[concept_idx], fontsize=12)
    plt.tight_layout()
    save_path = RUN_DIR / f"prototype_patches_concept_{concept_idx+1:02d}.png"
    plt.savefig(save_path, dpi=180, bbox_inches="tight")
    plt.show()
    return save_path


top_grounded = grounding_df["grounding_strength"].sort_values(ascending=False).index[:6].tolist()
prototype_plot_paths = [plot_concept_prototypes(idx) for idx in top_grounded]
sync_selected_to_drive([p.name for p in prototype_plot_paths])


In [ ]:
# ============================================================
# 17. Batch LLM explanations and export
# ============================================================
BATCH_EXPLAIN_DIR = RUN_DIR / "batch_explanations"
BATCH_EXPLAIN_DIR.mkdir(parents=True, exist_ok=True)


def safe_filename(text):
    text = re.sub(r"[^a-zA-Z0-9_\-]+", "_", str(text)).strip("_")
    return text[:120]


def explain_images_with_llm(model, image_paths, top_n=5):
    rows = []
    all_reports = []

    for i, path in enumerate(image_paths, 1):
        path = Path(path)
        stem = safe_filename(path.stem)
        case_id = f"{i:03d}_{stem}"
        fig_path = BATCH_EXPLAIN_DIR / f"{case_id}_concepts.png"
        evidence_path = BATCH_EXPLAIN_DIR / f"{case_id}_evidence.json"
        report_path = BATCH_EXPLAIN_DIR / f"{case_id}_doctor_report.md"

        print(f"Explaining {i}/{len(image_paths)}: {path.name}")
        exp = explain_image(model, path, top_n=top_n, save_path=fig_path, show_plot=False)

        with open(evidence_path, "w", encoding="utf-8") as f:
            json.dump(exp["llm_evidence_packet"], f, indent=2)

        report = None
        if USE_LLM_REPORTS:
            report = generate_llm_doctor_report(exp["llm_evidence_packet"])
            with open(report_path, "w", encoding="utf-8") as f:
                f.write(report)
            all_reports.append(f"\n\n---\n\n# Case {case_id}\n\n{report}\n")

        rows.append({
            "case_id": case_id,
            "image_path": str(path),
            "predicted_class": exp["predicted_class"],
            "confidence": exp["confidence"],
            "runner_up_class": exp["runner_up_class"],
            "runner_up_probability": exp["runner_up_probability"],
            "top_supporting_concepts": "; ".join([
                f"{r['concept_name']} ({r['margin_contribution']:+.4f})"
                for r in exp["llm_evidence_packet"]["main_supporting_margin_concepts"]
            ]),
            "figure_path": str(fig_path),
            "evidence_json": str(evidence_path),
            "llm_report_path": str(report_path) if report is not None else "",
        })

        rels = [str(fig_path.relative_to(RUN_DIR)), str(evidence_path.relative_to(RUN_DIR))]
        if report is not None:
            rels.append(str(report_path.relative_to(RUN_DIR)))
        sync_selected_to_drive(rels)

    df = pd.DataFrame(rows)
    summary_path = BATCH_EXPLAIN_DIR / "batch_explanation_summary.csv"
    df.to_csv(summary_path, index=False)

    if all_reports:
        combined_path = BATCH_EXPLAIN_DIR / "all_doctor_reports.md"
        with open(combined_path, "w", encoding="utf-8") as f:
            f.write("".join(all_reports))
    else:
        combined_path = None

    rels = [str(summary_path.relative_to(RUN_DIR))]
    if combined_path is not None:
        rels.append(str(combined_path.relative_to(RUN_DIR)))
    sync_selected_to_drive(rels)
    return df


selected = random.Random(SEED + 7).sample(all_test_images, min(BATCH_LLM_CASES, len(all_test_images)))
batch_summary_df = explain_images_with_llm(model, selected, top_n=5)
display(batch_summary_df)


In [ ]:
# ============================================================
# 18. Final metadata, weights, ZIP, and full Drive sync
# ============================================================
FINAL_WEIGHTS_PATH = RUN_DIR / "final_rg_cbm_effb1.weights.h5"
METADATA_PATH = RUN_DIR / "metadata.json"
model.save_weights(str(FINAL_WEIGHTS_PATH))

metadata = {
    "model_name": "RG-CBM EfficientNetB1",
    "run_name": RUN_NAME,
    "dataset_slug": DATASET_SLUG,
    "dataset_url": DATASET_URL,
    "dataset_root": str(DATASET_ROOT),
    "dataset_provenance_manifest": str(RUN_DIR / "dataset_provenance_manifest.json"),
    "train_dir": str(TRAIN_DIR),
    "test_dir": str(TEST_DIR),
    "class_names": class_names,
    "img_size": list(IMG_SIZE),
    "num_classes": num_classes,
    "num_concepts": NUM_CONCEPTS,
    "concept_groups": {
        "boundary": NUM_BOUNDARY_CONCEPTS,
        "texture": NUM_TEXTURE_CONCEPTS,
        "morphology": NUM_MORPHOLOGY_CONCEPTS,
        "unguided": NUM_UNGUIDED_CONCEPTS,
    },
    "concept_names": concept_names,
    "best_checkpoint": str(BEST_CKPT),
    "test_accuracy": float(accuracy_score(test_outputs["y_true"], test_outputs["y_pred"])),
    "llm_backend": LLM_BACKEND if USE_LLM_REPORTS else None,
    "llm_model": (GEMINI_MODEL if LLM_BACKEND == "gemini" else LOCAL_LLM_MODEL) if USE_LLM_REPORTS else None,
    "llm_fallback_model": LOCAL_LLM_MODEL if (USE_LLM_REPORTS and ALLOW_LOCAL_FALLBACK) else None,
    "llm_role": "Communication layer only; consumes structured RG-CBM output and does not affect prediction.",
    "manual_concept_annotations_used": False,
    "morphology_note": "Pseudo-morphology target; no manual segmentation masks required.",
    "texture_note": "Local variance + differentiable GLCM-inspired contrast/homogeneity proxy.",
    "faithfulness_note": "Both positive-support ablation and legacy absolute-contribution ablation are exported.",
}

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

ZIP_BASE = RUN_DIR.parent / f"{RUN_NAME}_results"
ZIP_PATH = Path(shutil.make_archive(
    str(ZIP_BASE), "zip",
    root_dir=str(RUN_DIR.parent),
    base_dir=RUN_DIR.name,
))

# Copy ZIP into the run folder too so the full Drive sync contains it.
ZIP_IN_RUN = RUN_DIR / ZIP_PATH.name
if ZIP_PATH.resolve() != ZIP_IN_RUN.resolve():
    shutil.copy2(ZIP_PATH, ZIP_IN_RUN)

sync_selected_to_drive(sync_all=True)

print("Final weights:", FINAL_WEIGHTS_PATH)
print("Metadata:", METADATA_PATH)
print("ZIP:", ZIP_IN_RUN)
if DRIVE_RUN_DIR is not None:
    print("Google Drive copy:", DRIVE_RUN_DIR)
